# Project FORESIGHT
## Inventory Risk Scoring & Recommended Actions

### Objective

Combine demand forecasts with inventory information to identify SKU-level
inventory risks and recommend appropriate actions.

The notebook will classify SKUs into:

- Stockout Risk
- Overstock Risk
- Healthy

It will also generate recommended inventory actions.

In [15]:
import pandas as pd
import numpy as np

from pathlib import Path

In [16]:
ROOT_DIR = Path.cwd().parent

PROCESSED_DIR = ROOT_DIR / "data" / "processed"

forecast = pd.read_csv(
    PROCESSED_DIR / "forecast_predictions.csv"
)

forecast["week_start"] = pd.to_datetime(
    forecast["week_start"]
)

print("Forecast shape:", forecast.shape)

forecast.head()

Forecast shape: (2400, 4)


,week_start,sku_id,units_sold,prediction
0,2025-10-13,SKU_0001,16,21.475205
1,2025-10-20,SKU_0001,22,27.584302
2,2025-10-27,SKU_0001,26,28.126664
3,2025-11-03,SKU_0001,31,24.259666
4,2025-11-10,SKU_0001,26,22.289730


In [17]:
inventory = pd.read_csv(
    ROOT_DIR / "data" / "raw" / "inventory_snapshots.csv"
)

inventory["date"] = pd.to_datetime(
    inventory["date"]
)

print("Inventory shape:", inventory.shape)

inventory.head()

Inventory shape: (31400, 6)


,date,sku_id,on_hand_units,on_order_units,lead_time_days,reorder_point
0,2023-01-01,SKU_0001,64,0,5,17
1,2023-01-08,SKU_0001,45,0,5,17
2,2023-01-15,SKU_0001,30,0,5,17
3,2023-01-22,SKU_0001,0,36,5,17
4,2023-01-29,SKU_0001,36,0,5,17


In [18]:
latest_inventory = (
    inventory
    .sort_values("date")
    .groupby("sku_id")
    .tail(1)
    .copy()
)

latest_inventory = latest_inventory[
    [
        "sku_id",
        "date",
        "on_hand_units",
        "on_order_units",
        "lead_time_days",
        "reorder_point"
    ]
]

latest_inventory.head()

,sku_id,date,on_hand_units,on_order_units,lead_time_days,reorder_point
31242,SKU_0199,2025-12-28,214,0,8,160
15699,SKU_0100,2025-12-28,77,272,4,108
30928,SKU_0197,2025-12-28,265,0,3,76
10989,SKU_0070,2025-12-28,39,76,10,61
10832,SKU_0069,2025-12-28,38,252,13,186


In [19]:
forecast_summary = (
    forecast
    .groupby("sku_id")
    .agg(
        forecast_12w=("prediction", "sum"),
        avg_weekly_forecast=("prediction", "mean")
    )
    .reset_index()
)

forecast_summary.head()

,sku_id,forecast_12w,avg_weekly_forecast
0,SKU_0001,313.474140,26.122845
1,SKU_0002,450.875042,37.572920
2,SKU_0003,301.584112,25.132009
3,SKU_0004,931.043867,77.586989
4,SKU_0005,1638.156567,136.513047


In [20]:
risk_df = forecast_summary.merge(
    latest_inventory,
    on="sku_id",
    how="left"
)

risk_df.head()

,sku_id,forecast_12w,avg_weekly_forecast,date,on_hand_units,on_order_units,lead_time_days,reorder_point
0,SKU_0001,313.474140,26.122845,2025-12-28,16,44,5,17
1,SKU_0002,450.875042,37.572920,2025-12-28,66,0,14,54
2,SKU_0003,301.584112,25.132009,2025-12-28,36,0,6,19
3,SKU_0004,931.043867,77.586989,2025-12-28,0,121,9,97
4,SKU_0005,1638.156567,136.513047,2025-12-28,94,241,9,156


In [21]:
risk_df["weeks_of_cover"] = np.where(
    risk_df["avg_weekly_forecast"] > 0,
    (
        risk_df["on_hand_units"]
        + risk_df["on_order_units"]
    )
    / risk_df["avg_weekly_forecast"],
    np.inf
)

risk_df["weeks_of_cover"] = risk_df[
    "weeks_of_cover"
].round(2)

risk_df.head()

,sku_id,forecast_12w,avg_weekly_forecast,date,on_hand_units,on_order_units,lead_time_days,reorder_point,weeks_of_cover
0,SKU_0001,313.474140,26.122845,2025-12-28,16,44,5,17,2.30
1,SKU_0002,450.875042,37.572920,2025-12-28,66,0,14,54,1.76
2,SKU_0003,301.584112,25.132009,2025-12-28,36,0,6,19,1.43
3,SKU_0004,931.043867,77.586989,2025-12-28,0,121,9,97,1.56
4,SKU_0005,1638.156567,136.513047,2025-12-28,94,241,9,156,2.45


In [22]:
risk_df["stockout_risk"] = np.where(
    risk_df["on_hand_units"] <= risk_df["reorder_point"],
    1,
    0
)

In [23]:
risk_df["overstock_risk"] = np.where(
    risk_df["on_hand_units"]
    > risk_df["forecast_12w"],
    1,
    0
)

In [24]:
risk_df["risk_category"] = np.select(
    [
        (
            (risk_df["on_hand_units"] == 0)
            |
            (
                risk_df["weeks_of_cover"]
                < risk_df["lead_time_days"] / 7
            )
        ),

        (
            risk_df["on_hand_units"]
            > risk_df["forecast_12w"]
        )
    ],
    [
        "Stockout Risk",
        "Overstock Risk"
    ],
    default="Healthy"
)

risk_df["risk_category"].value_counts()

risk_category
Healthy          172
Stockout Risk     28
Name: count, dtype: int64

In [25]:
risk_df["recommended_action"] = np.select(
    [
        risk_df["risk_category"] == "Stockout Risk",

        risk_df["risk_category"] == "Overstock Risk"
    ],
    [
        "Prioritize replenishment",

        "Reduce or defer replenishment"
    ],
    default="Maintain current inventory"
)

risk_df[
    [
        "sku_id",
        "risk_category",
        "recommended_action"
    ]
].head(20)

,sku_id,risk_category,recommended_action
0,SKU_0001,Healthy,Maintain current inventory
1,SKU_0002,Stockout Risk,Prioritize replenishment
2,SKU_0003,Healthy,Maintain current inventory
3,SKU_0004,Stockout Risk,Prioritize replenishment
4,SKU_0005,Healthy,Maintain current inventory
5,SKU_0006,Healthy,Maintain current inventory
6,SKU_0007,Healthy,Maintain current inventory
7,SKU_0008,Healthy,Maintain current inventory
8,SKU_0009,Healthy,Maintain current inventory
9,SKU_0010,Healthy,Maintain current inventory


In [26]:
risk_summary = (
    risk_df["risk_category"]
    .value_counts()
    .rename_axis("risk_category")
    .reset_index(name="sku_count")
)

risk_summary

,risk_category,sku_count
0,Healthy,172
1,Stockout Risk,28


In [27]:
priority_skus = (
    risk_df
    .sort_values(
        [
            "risk_category",
            "weeks_of_cover"
        ],
        ascending=[True, True]
    )
)

priority_skus[
    [
        "sku_id",
        "forecast_12w",
        "on_hand_units",
        "on_order_units",
        "lead_time_days",
        "reorder_point",
        "weeks_of_cover",
        "risk_category",
        "recommended_action"
    ]
].head(20)

,sku_id,forecast_12w,on_hand_units,on_order_units,lead_time_days,reorder_point,weeks_of_cover,risk_category,recommended_action
137,SKU_0138,2911.160338,105,0,3,85,0.43,Healthy,Maintain current inventory
189,SKU_0190,705.517622,34,0,3,20,0.58,Healthy,Maintain current inventory
136,SKU_0137,2794.306188,172,0,5,141,0.74,Healthy,Maintain current inventory
131,SKU_0132,1760.253988,112,0,4,81,0.76,Healthy,Maintain current inventory
163,SKU_0164,2230.139858,146,0,5,123,0.79,Healthy,Maintain current inventory
30,SKU_0031,1785.499253,122,0,4,77,0.82,Healthy,Maintain current inventory
139,SKU_0140,461.784820,35,0,4,19,0.91,Healthy,Maintain current inventory
33,SKU_0034,4275.122740,340,0,5,242,0.95,Healthy,Maintain current inventory
57,SKU_0058,555.588841,45,0,4,22,0.97,Healthy,Maintain current inventory
67,SKU_0068,2516.161399,212,0,6,148,1.01,Healthy,Maintain current inventory


In [28]:
output_path = (
    PROCESSED_DIR
    / "inventory_risk_scores.csv"
)

risk_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(risk_df))

Saved: c:\Users\saqib\Desktop\zidio\foresight-demand-inventory\data\processed\inventory_risk_scores.csv
Rows: 200


# Conclusion

The inventory risk layer combines forecasted demand with current inventory,
on-order inventory, lead time and reorder points.

Each SKU receives:

- Forecasted 12-week demand
- Weeks of inventory cover
- Stockout risk
- Overstock risk
- Risk category
- Recommended action

The resulting `inventory_risk_scores.csv` will be used by the decision-support
dashboard and recommendation layer.